In [28]:
import pymysql
import urllib.parse
from sqlalchemy import create_engine
import pandas as pd
import ast

DB_USER = "root"
DB_PASS = "sadegh24"
DB_HOST = "localhost"
DB_PORT = "3306"
DB_NAME = "ashpaz"

safe_pass = urllib.parse.quote_plus(DB_PASS)

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{safe_pass}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

conn = pymysql.connect(
    host=DB_HOST, 
    user=DB_USER, 
    password=DB_PASS, 
    database=DB_NAME, 
    autocommit=True
)
cursor = conn.cursor()

In [29]:
cursor.execute("SET max_execution_time = 0;")
cursor.execute("SET FOREIGN_KEY_CHECKS = 0;")

tables_to_drop = ["menu_item", "restaurant_cuisine", "cuisine", "restaurant", "services", "location"]

for table in tables_to_drop:
    cursor.execute(f"DROP TABLE IF EXISTS {table};")
    
cursor.execute("SET FOREIGN_KEY_CHECKS = 1;")

0

In [30]:
cursor.execute("""
CREATE TABLE location (
    id INT AUTO_INCREMENT PRIMARY KEY,
    city_name VARCHAR(255) NOT NULL,
    area_name VARCHAR(255) NOT NULL,
    UNIQUE(city_name, area_name)
)
""")

cursor.execute("""
CREATE TABLE cuisine (
    id INT AUTO_INCREMENT PRIMARY KEY,
    cuisine_name VARCHAR(255) NOT NULL UNIQUE
)
""")

cursor.execute("""
CREATE TABLE services (
    id INT PRIMARY KEY,
    online_order VARCHAR(50),
    book_table VARCHAR(50),
    rate VARCHAR(50),
    votes INT,
    listed_in_type VARCHAR(255)
)
""")

cursor.execute("""
CREATE TABLE restaurant (
    id INT PRIMARY KEY,
    name VARCHAR(255) NOT NULL,
    address TEXT,
    phone TEXT,
    approx_cost INT,
    rest_type VARCHAR(255),
    location_id INT,
    service_id INT,
    FOREIGN KEY (location_id) REFERENCES location(id) ON DELETE SET NULL,
    FOREIGN KEY (service_id) REFERENCES services(id) ON DELETE SET NULL
)
""")

cursor.execute("""
CREATE TABLE restaurant_cuisine (
    restaurant_id INT,
    cuisine_id INT,
    PRIMARY KEY (restaurant_id, cuisine_id),
    FOREIGN KEY (restaurant_id) REFERENCES restaurant(id) ON DELETE CASCADE,
    FOREIGN KEY (cuisine_id) REFERENCES cuisine(id) ON DELETE CASCADE
)
""")

cursor.execute("""
CREATE TABLE menu_item (
    id INT AUTO_INCREMENT PRIMARY KEY,
    restaurant_id INT,
    item_name VARCHAR(255) NOT NULL,
    FOREIGN KEY (restaurant_id) REFERENCES restaurant(id) ON DELETE CASCADE
)
""")
conn.commit()

In [31]:
locations_set = set()
cuisines_set = set()
services_set = set()
all_rows_parsed = []

with open('zomato.csv', 'r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        name = row.get('name', '').strip()
        address = row.get('address', '').strip()
        if not name or not address:
            continue

        city = row.get('listed_in(city)', '').strip()
        area = row.get('location', '').strip()
        if city and area:
            locations_set.add((city, area))

        c_str = row.get('cuisines', '')
        if c_str:
            for c in c_str.split(','):
                if c.strip():
                    cuisines_set.add(c.strip())

        online_order = row.get('online_order', '')
        book_table = row.get('book_table', '')
        rate = row.get('rate', '')
        
        try:
            votes = int(row.get('votes', 0) or 0)
        except:
            votes = 0

        listed_in_type = row.get('listed_in(type)', '')
        
        service_tuple = (online_order, book_table, rate, votes, listed_in_type)
        services_set.add(service_tuple)

        all_rows_parsed.append({
            'name': name, 'address': address, 'phone': row.get('phone', '').strip(),
            'rest_type': row.get('rest_type', '').strip(),
            'approx_cost': row.get('approx_cost(for two people)', '').replace(',', '').strip(),
            'city': city, 'area': area, 'cuisines': c_str,
            'menu_item': row.get('menu_item', ''), 'service_tuple': service_tuple
        })

location_map = {val: idx for idx, val in enumerate(locations_set, 1)}
cuisine_map = {val: idx for idx, val in enumerate(cuisines_set, 1)}
service_map = {val: idx for idx, val in enumerate(services_set, 1)}

loc_insert_data = [(v, k[0], k[1]) for k, v in location_map.items()]
cuis_insert_data = [(v, k) for k, v in cuisine_map.items()]

srv_insert_data = [(v, k[0], k[1], k[2], k[3], k[4]) for k, v in service_map.items()]

cursor.executemany("INSERT INTO location (id, city_name, area_name) VALUES (%s, %s, %s)", loc_insert_data)
cursor.executemany("INSERT INTO cuisine (id, cuisine_name) VALUES (%s, %s)", cuis_insert_data)
cursor.executemany("INSERT INTO services (id, online_order, book_table, rate, votes, listed_in_type) VALUES (%s, %s, %s, %s, %s, %s)", srv_insert_data)

rest_insert_data = []
rest_cuisines_data = set()
menu_items_data = []

for idx, data in enumerate(all_rows_parsed, 1):
    loc_id = location_map.get((data['city'], data['area']))
    srv_id = service_map.get(data['service_tuple'])

    try:
        approx_cost = int(data['approx_cost'])
    except:
        approx_cost = None

    rest_insert_data.append((idx, data['name'], data['address'], data['phone'], approx_cost, data['rest_type'], loc_id, srv_id))

    if data['cuisines']:
        for c in data['cuisines'].split(','):
            c_id = cuisine_map.get(c.strip())
            if c_id:
                rest_cuisines_data.add((idx, c_id))

    m_str = data['menu_item']
    if m_str and m_str != '[]':
        try:
            menu_list = ast.literal_eval(m_str)
            if isinstance(menu_list, list):
                for item in menu_list:
                    if isinstance(item, str) and item.strip():
                        menu_items_data.append((idx, item.strip()[:255]))
        except:
            pass

chunk_size = 10000
for i in range(0, len(rest_insert_data), chunk_size):
    cursor.executemany("""
        INSERT INTO restaurant (id, name, address, phone, approx_cost, rest_type, location_id, service_id)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
    """, rest_insert_data[i:i+chunk_size])

for i in range(0, len(list(rest_cuisines_data)), chunk_size):
    cursor.executemany("INSERT IGNORE INTO restaurant_cuisine (restaurant_id, cuisine_id) VALUES (%s, %s)", list(rest_cuisines_data)[i:i+chunk_size])

for i in range(0, len(menu_items_data), chunk_size):
    cursor.executemany("INSERT INTO menu_item (restaurant_id, item_name) VALUES (%s, %s)", menu_items_data[i:i+chunk_size])

conn.commit()

cursor.execute("SELECT COUNT(*) FROM services")
print(f"Total unique services inserted: {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM restaurant")
print(f"Total restaurants inserted: {cursor.fetchone()[0]}")

Total unique services inserted: 6274
Total restaurants inserted: 12429


In [32]:
queries = {
    "1. Delivery Restaurants Stats": """
        SELECT 
            COUNT(r.id) AS total_restaurants,
            AVG(r.approx_cost) AS average_cost
        FROM restaurant r
        JOIN services s ON r.service_id = s.id
        WHERE s.listed_in_type = 'Delivery';
    """,
    
    "2. Online Order Impact on Pricing": """
        SELECT 
            l.city_name,
            s.online_order,
            AVG(r.approx_cost) AS average_cost,
            SUM(s.votes) AS total_votes
        FROM restaurant r
        JOIN services s ON r.service_id = s.id
        JOIN location l ON r.location_id = l.id
        WHERE s.online_order IN ('Yes', 'No')
        GROUP BY l.city_name, s.online_order
        ORDER BY l.city_name ASC, s.online_order DESC;
    """,
    
    "3. Top-tier Neighborhoods (Rate > 4.2 & Count >= 50)": """
        SELECT 
            l.area_name AS neighborhood,
            AVG(
                CASE 
                    WHEN s.rate REGEXP '^[0-9]' THEN CAST(SUBSTRING_INDEX(s.rate, '/', 1) AS DECIMAL(3,1)) 
                    ELSE NULL 
                END
            ) AS average_rating,
            COUNT(r.id) AS restaurant_count
        FROM restaurant r
        JOIN services s ON r.service_id = s.id
        JOIN location l ON r.location_id = l.id
        GROUP BY l.area_name
        HAVING average_rating > 4.2 AND restaurant_count >= 50
        ORDER BY average_rating DESC;
    """,
    
    "4. Pricing Tier Dashboard": """
        SELECT 
            l.city_name AS city,
            CASE 
                WHEN r.approx_cost <= 500 THEN 'Budget'
                WHEN r.approx_cost <= 1000 THEN 'Mid-Range'
                ELSE 'Premium'
            END AS tier,
            COUNT(r.id) AS restaurant_count,
            AVG(
                CASE 
                    WHEN s.rate REGEXP '^[0-9]' THEN CAST(SUBSTRING_INDEX(s.rate, '/', 1) AS DECIMAL(3,1)) 
                    ELSE NULL 
                END
            ) AS average_rating
        FROM restaurant r
        JOIN services s ON r.service_id = s.id
        JOIN location l ON r.location_id = l.id
        WHERE r.approx_cost IS NOT NULL
        GROUP BY l.city_name, tier
        ORDER BY l.city_name ASC, restaurant_count DESC;
    """
}

for title, query in queries.items():
    print("\n")
    print(f"Executing: {title}")
    print("\n")
    
    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        
        columns = [desc[0] for desc in cursor.description]
        
        df = pd.DataFrame(rows, columns=columns)
        
        if df.empty:
            print("No data returned for this query.")
        else:
            display(df) 
            
    except Exception as e:
        print(f"Error executing query: {e}")

cursor.close()
conn.close()



Executing: 1. Delivery Restaurants Stats




,total_restaurants,average_cost
0,8674,433.8652




Executing: 2. Online Order Impact on Pricing




,city_name,online_order,average_cost,total_votes
0,Banashankari,Yes,420.2101,82291
1,Banashankari,No,336.8326,19183
2,Bannerghatta Road,Yes,430.6140,111558
3,Bannerghatta Road,No,420.5641,29029
4,Basavanagudi,Yes,450.9756,41315
5,Basavanagudi,No,390.6417,23810
6,Bellandur,Yes,496.3109,125780
7,Bellandur,No,508.8142,31742
8,Brigade Road,Yes,575.6571,167818
9,Brigade Road,No,822.2222,136681




Executing: 3. Top-tier Neighborhoods (Rate > 4.2 & Count >= 50)


No data returned for this query.


Executing: 4. Pricing Tier Dashboard




,city,tier,restaurant_count,average_rating
0,Banashankari,Budget,465,3.62324
1,Banashankari,Mid-Range,121,3.75603
2,Banashankari,Premium,11,3.95455
3,Bannerghatta Road,Budget,826,3.51274
4,Bannerghatta Road,Mid-Range,219,3.58936
...,...,...,...,...
80,Sarjapur Road,Mid-Range,18,3.49375
81,Sarjapur Road,Premium,4,3.92500
82,Whitefield,Budget,335,3.45931
83,Whitefield,Mid-Range,87,3.64225
